# 08 - Test-Time Augmentation
Compare four deterministic test-time augmentation presets on a fixed checkpoint. The model is fixed; only inference-time TTA changes.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
from src.models import load_arcface_model_from_checkpoint
from src.training import compute_map_from_embeddings
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device


## Base Config


In [ ]:
config = {
    "data_dir": Path("../data"),
    "results_dir": Path("../checkpoints/e08_tta_comparison"),
    "submission_dir": Path("../submissions/e08_tta_comparison"),
    "batch_size": 32,
    "num_workers": 2,
    "save_submissions": True,
    "seed": RANDOM_SEED,
}

config["results_dir"].mkdir(parents=True, exist_ok=True)
config["submission_dir"].mkdir(parents=True, exist_ok=True)
config


## Checkpoint


In [ ]:
checkpoint_specs = [
    {
        "name": "eva02_finetune_seed42",
        "path": Path("../checkpoints/e2_backbone_finetune/EVA02_Large_finetune.pth"),
    },
]

checkpoint_specs


## TTA Presets


In [ ]:
tta_presets = {
    "none": [
        {"name": "base", "crop_scale": 1.00, "anchor": "center"},
    ],
    "light": [
        {"name": "base", "crop_scale": 1.00, "anchor": "center"},
        {"name": "center_95", "crop_scale": 0.95, "anchor": "center"},
    ],
    "medium": [
        {"name": "base", "crop_scale": 1.00, "anchor": "center"},
        {"name": "center_95", "crop_scale": 0.95, "anchor": "center"},
        {"name": "center_90", "crop_scale": 0.90, "anchor": "center"},
    ],
    "heavy": [
        {"name": "base", "crop_scale": 1.00, "anchor": "center"},
        {"name": "center_95", "crop_scale": 0.95, "anchor": "center"},
        {"name": "center_90", "crop_scale": 0.90, "anchor": "center"},
        {"name": "top_left_90", "crop_scale": 0.90, "anchor": "top_left"},
        {"name": "bottom_right_90", "crop_scale": 0.90, "anchor": "bottom_right"},
    ],
}

tta_presets


## Run TTA Comparison


In [ ]:
results = []

for checkpoint_spec in checkpoint_specs:
    checkpoint_path = checkpoint_spec["path"]
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    print("=" * 90)
    print(f"Checkpoint: {checkpoint_spec['name']} -> {checkpoint_path}")

    model, checkpoint, checkpoint_config = load_arcface_model_from_checkpoint(checkpoint_path, device)
    eval_frames = data.build_eval_frames_from_config(checkpoint_config)
    rerank_config = checkpoint_config.get("rerank", {"enabled": False, "k1": 20, "k2": 6, "lambda_value": 0.3})
    input_size = int(checkpoint_config["input_size"])
    batch_size = int(checkpoint_config.get("batch_size", config["batch_size"]))
    num_workers = int(checkpoint_config.get("num_workers", config["num_workers"]))

    for tta_name, views in tta_presets.items():
        print(f"\nTTA mode: {tta_name} | views={[view['name'] for view in views]}")

        val_embeddings, val_labels = inference.extract_tta_embeddings_with_labels(
            model,
            eval_frames["val_df"],
            eval_frames["data_dir"] / "train" / "train",
            input_size,
            batch_size,
            num_workers,
            views,
            device,
            seed=int(config["seed"]),
        )
        val_map = compute_map_from_embeddings(val_embeddings, val_labels)
        val_map_rerank = compute_map_from_embeddings(
            val_embeddings,
            val_labels,
            use_rerank=rerank_config.get("enabled", False),
            k1=rerank_config.get("k1", 20),
            k2=rerank_config.get("k2", 6),
            lambda_value=rerank_config.get("lambda_value", 0.3),
        )

        test_names, test_embeddings = inference.extract_tta_embeddings_with_names(
            model,
            eval_frames["test_df"],
            eval_frames["data_dir"] / "test" / "test",
            input_size,
            batch_size,
            num_workers,
            views,
            device,
            seed=int(config["seed"]),
        )

        plain_submission_path = config["submission_dir"] / f"{checkpoint_spec['name']}__tta_{tta_name}.csv"
        rerank_submission_path = config["submission_dir"] / f"{checkpoint_spec['name']}__tta_{tta_name}_rerank.csv"
        if config["save_submissions"]:
            inference.create_submission_from_embeddings(
                eval_frames["pairs_df"],
                test_names,
                test_embeddings,
                output_path=plain_submission_path,
                use_rerank=False,
            )
            inference.create_submission_from_embeddings(
                eval_frames["pairs_df"],
                test_names,
                test_embeddings,
                output_path=rerank_submission_path,
                use_rerank=rerank_config.get("enabled", False),
                k1=rerank_config.get("k1", 20),
                k2=rerank_config.get("k2", 6),
                lambda_value=rerank_config.get("lambda_value", 0.3),
            )

        results.append(
            {
                "checkpoint_name": checkpoint_spec["name"],
                "checkpoint_path": str(checkpoint_path),
                "seed": int(checkpoint_config["seed"]),
                "tta_mode": tta_name,
                "tta_views": ",".join(view["name"] for view in views),
                "tta_view_count": len(views),
                "val_map": val_map,
                "val_map_rerank": val_map_rerank,
                "plain_submission_path": str(plain_submission_path) if config["save_submissions"] else None,
                "rerank_submission_path": str(rerank_submission_path) if config["save_submissions"] else None,
            }
        )

results_df = pd.DataFrame(results).sort_values(["val_map_rerank", "val_map"], ascending=False).reset_index(drop=True)
results_path = config["results_dir"] / "tta_comparison_results.csv"
results_df.to_csv(results_path, index=False)
results_df


## Summary


In [ ]:
results_df[[
    "checkpoint_name",
    "seed",
    "tta_mode",
    "tta_view_count",
    "val_map",
    "val_map_rerank",
]].sort_values(["val_map_rerank", "val_map"], ascending=False)


View the results on W&B: [W&B Run Group](https://wandb.ai/juggling-jaguars/jaguar-reid-jugglingjaguars/groups/Experiment-8-TestTimeAugmentation)